# Cyberbully Shield: Real-Time Inference

Pass raw test strings and image paths to the trained multimodal model to evaluate real-time capability.

In [2]:
import os
import sys
import torch
from PIL import Image
from torchvision import transforms
from transformers import DistilBertTokenizer

sys.path.append(os.path.abspath('../src'))
from model import CyberbullyShieldFusion
from data_ingestion import clean_text, extract_text_from_image


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CyberbullyShieldFusion().to(device)
model.load_state_dict(torch.load('../checkpoints/best_shield_model.pth', map_location=device))
model.eval()

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def predict_cyberbullying(text="", image_path=None):
    # 1. Process Image
    image_tensor = None
    extracted_text = ""
    if image_path and os.path.exists(image_path):
        try:
            img = Image.open(image_path).convert("RGB")
            image_tensor = transform(img).unsqueeze(0).to(device)
            extracted_text = extract_text_from_image(image_path)
        except Exception as e:
            print(f"Error processing image: {e}")
            image_tensor = torch.zeros((1, 3, 224, 224)).to(device)
    else:
        image_tensor = torch.zeros((1, 3, 224, 224)).to(device)
        
    # 2. Process Text
    combined_text = f"{text} {extracted_text}".strip()
    cleaned_text = clean_text(combined_text)
    
    if not cleaned_text:
        cleaned_text = ""
        
    encoding = tokenizer(
        cleaned_text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # 3. Model Inference
    with torch.no_grad():
        logits = model(input_ids, attention_mask, image_tensor)
        prob = torch.sigmoid(logits).item()
        
    prediction = "Bullying" if prob > 0.5 else "Non-Bullying"
    print("="*40)
    print(f"Input Text: '{text}'")
    print(f"Image Path: '{image_path}'")
    print(f"Processed Combined Text: '{cleaned_text}'")
    print(f"Prediction: {prediction} (Confidence: {prob:.4f})")
    print("="*40)
    return prob, prediction

In [8]:
# Example Usage:
predict_cyberbullying(text="You are such a loser, nobody likes you!!!", image_path="")
predict_cyberbullying(text="Had a great day at the park today. #happy")
predict_cyberbullying(text="", image_path="D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\5568.jpg")
predict_cyberbullying(text="A good article, which, for all the good it will do, could have been delivered to a brick wall.")

Input Text: 'You are such a loser, nobody likes you!!!'
Image Path: ''
Processed Combined Text: 'You are such a loser, nobody likes you!!!'
Prediction: Bullying (Confidence: 0.9989)
Input Text: 'Had a great day at the park today. #happy'
Image Path: 'None'
Processed Combined Text: 'Had a great day at the park today. happy'
Prediction: Non-Bullying (Confidence: 0.0001)
Input Text: ''
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_dataŮ8.jpg'
Processed Combined Text: ''
Prediction: Non-Bullying (Confidence: 0.0017)
Input Text: 'A good article, which, for all the good it will do, could have been delivered to a brick wall.'
Image Path: 'None'
Processed Combined Text: 'A good article, which, for all the good it will do, could have been delivered to a brick wall.'
Prediction: Non-Bullying (Confidence: 0.0003)


(0.0002651906106621027, 'Non-Bullying')

In [6]:
import pandas as pd
from tqdm import tqdm # For a nice progress bar!

# 1. Load the CSV file
csv_path = "D:/THE_CYBERBULLY_SHIELD/Datasets" 
df = pd.read_csv(csv_path)

# Verify the columns exist (adjust 'text' and 'label' if your CSV header names are different)
text_column = 'Text' 
label_column = 'oh_label'

if text_column not in df.columns or label_column not in df.columns:
    print(f"Error: Could not find columns '{text_column}' and/or '{label_column}' in CSV.")
else:
    # 2. Randomly sample exactly 500 rows
    sample_size = min(100, len(df)) # In case the CSV has less than 500 rows
    test_df = df.sample(n=sample_size)
    
    correct_predictions = 0
    total_predictions = 0
    
    print(f"--- Starting Evaluation on {sample_size} Random Samples ---\n")
    
    # 3. Loop through the 500 rows
    for index, row in tqdm(test_df.iterrows(), total=sample_size, desc="Predicting"):
        raw_text = str(row[text_column])
        true_label = row[label_column] 
        
        # 4. Pass the text to your model's function (image_path is None since this is a text CSV)
        # Note: We capture the printed output of predict_cyberbullying, but we only really need the returned values
        prob, prediction_string = predict_cyberbullying(text=raw_text, image_path=None)
        
        # 5. Print the comparison
        # (Comment out the line below if you don't want it to spam your notebook with 500 print statements!)
        
        # print(f"\nText: '{raw_text[:80]}...' | \nModel Said: {prediction_string} | \nTrue Label: {true_label}\n")
        
        # Optional: Calculate a quick accuracy score
        # Assuming your CSV label is 1 for Bullying and 0 for Non-Bullying
        is_model_bullying = (prediction_string == "Bullying")
        is_true_bullying = (str(true_label) == '1' or str(true_label).lower() == 'bullying')
        
        if is_model_bullying == is_true_bullying:
            correct_predictions += 1
            
        total_predictions += 1
        
    # Final Results
    accuracy = (correct_predictions / total_predictions) * 100
    print(f"\n{'='*40}")
    print(f"FINISHED! Tested {total_predictions} samples.")
    print(f"Overall Accuracy on this CSV subset: {accuracy:.2f}%")
    print(f"{'='*40}")


PermissionError: [Errno 13] Permission denied: 'D:/THE_CYBERBULLY_SHIELD/Datasets'

In [8]:
import pandas as pd
import os
from tqdm import tqdm

# 1. Define paths for the CSV and the Image Folder
csv_path = "D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/preprocessed_cyberbully.csv" 
image_folder = "D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data"

# Load the CSV
df = pd.read_csv(csv_path)

# 2. Map your CSV columns
image_name_col = 'Img-Name'
text_col = 'Img-Text'
img_label_col = 'Img-Label' 
text_label_col = 'Text-Label' 

# Verify the columns exist
if not all(col in df.columns for col in [image_name_col, img_label_col, text_label_col]):
    print("Error: Could not find one or more required columns in the CSV.")
else:
    # 3. Randomly sample x rows (set to 20)
    sample_size = min(20, len(df)) 
    test_df = df.sample(n=sample_size)
    
    # Trackers for our two different metrics
    correct_img_predictions = 0
    correct_text_predictions = 0
    total_valid_predictions = 0
    
    print(f"--- Starting Multimodal Evaluation on {sample_size} Random Samples ---\n")
    
    # 4. Loop through the sampled rows
    for index, row in tqdm(test_df.iterrows(), total=sample_size, desc="Predicting"):
        raw_text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        img_name = str(row[image_name_col])
        
        true_img_label = row[img_label_col]
        true_text_label = row[text_label_col]
        
        # Format the image path safely
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_name += '.jpg'
            
        full_image_path = os.path.join(image_folder, img_name)
        
        # --- NEW: Check if the image actually exists before predicting ---
        if not os.path.exists(full_image_path):
            print(f"\n[WARNING] Skipping {img_name} - File not found at {full_image_path}")
            continue # Skips to the next row in the loop
            
        # 5. Pass data to the model
        prob, prediction_string = predict_cyberbullying(text=raw_text, image_path=full_image_path)
        
        
        # 6. Print the comparison
        print(f"\nImage: {img_name} | Text: '{raw_text[:40]}...'")
        print(f"Model Said: {prediction_string} | True Img Label: {true_img_label} | True Text Label: {true_text_label}\n")
        
        # 7. Evaluate against BOTH labels
        is_model_bullying = (prediction_string == "Bullying")
        
        is_true_img_bullying = (str(true_img_label) == '1' or str(true_img_label).lower() == 'bullying')
        is_true_text_bullying = (str(true_text_label) == '1' or str(true_text_label).lower() == 'bullying')
        
        if is_model_bullying == is_true_img_bullying:
            correct_img_predictions += 1
            
        if is_model_bullying == is_true_text_bullying:
            correct_text_predictions += 1
            
        total_valid_predictions += 1
        
    # Final Results
    if total_valid_predictions > 0:
        img_accuracy = (correct_img_predictions / total_valid_predictions) * 100
        text_accuracy = (correct_text_predictions / total_valid_predictions) * 100
        
        print(f"\n{'='*50}")
        print(f"FINISHED! Tested {total_valid_predictions} valid image samples.")
        print(f"Accuracy vs Img-Label:  {img_accuracy:.2f}%")
        print(f"Accuracy vs Text-Label: {text_accuracy:.2f}%")
        print(f"{'='*50}")
    else:
        print("\nNo valid predictions were made (perhaps all images were missing?).")

--- Starting Multimodal Evaluation on 20 Random Samples ---



Predicting:   5%|▌         | 1/20 [00:02<00:51,  2.74s/it]

Input Text: 'Aur Daalo Thoda Aur Daalo Itna thik hai Chini, Chai mein'
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\441.jpg'
Processed Combined Text: 'Aur Daalo Thoda Aur Daalo Itna thik hai Chini, Chai mein Aur Daalo Thoda Aur Daalo Itna thik hai Chini; chbitclln'
Prediction: Non-Bullying (Confidence: 0.4375)

Image: 441.jpg | Text: 'Aur Daalo Thoda Aur Daalo Itna thik hai ...'
Model Said: Non-Bullying | True Img Label: 0 | True Text Label: 0



Predicting:  15%|█▌        | 3/20 [00:03<00:14,  1.21it/s]

Input Text: 'what the people deserve: the right to love who they want the right to equal pay and respect the right to become a citicen and work hard like the rest'
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\1078.png'
Processed Combined Text: 'what the people deserve: the right to love who they want the right to equal pay and respect the right to become a citicen and work hard like the rest wbaiubpdeqi2 IustuR Ud mur mmimmwrtr Mhc myloco AITaI TrW TEeesmecI IImDI ^ _Pcome ; CILICE aML xmziMadMzp Iif2e51'
Prediction: Non-Bullying (Confidence: 0.0002)

Image: 1078.png | Text: 'what the people deserve: the right to lo...'
Model Said: Non-Bullying | True Img Label: 0 | True Text Label: 0

Input Text: '"Me turns on my TV"Le naaptol employees for no reason:'
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\3197.jpg'
Processed Combined Text: '"Me turns on my TV"Le naaptol employees for no reason: He Turns Hnapto Employco

...

Input Text: 'Look,man...I got a masters in electrical engineering Welcome to BASKIN-ROBBINS The most realistic scene in any MARVEL movie'
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\4158.jpg'
Processed Combined Text: 'Look,man...I got a masters in electrical engineering Welcome to BASKIN-ROBBINS The most realistic scene in any MARVEL movie Ook, MAN dgovalesters IN ELECTRICAL ENGINEERING cbil WWELCOME TO BASKIN-ROBBINS Tho Most Reallatic scone any MARVEL movie'
Prediction: Non-Bullying (Confidence: 0.0047)

Image: 4158.jpg | Text: 'Look,man...I got a masters in electrical...'
Model Said: Non-Bullying | True Img Label: 0 | True Text Label: 1



Predicting:  95%|█████████▌| 19/20 [00:06<00:00,  4.34it/s]

Input Text: 'DID YOU KNOW? Chocolate improves brain function Banana reduces depression Orange improves skin health Cucumber keeps you hydrated  FITTYWITTYOFFICIAL'
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\5512.jpg'
Processed Combined Text: 'DID YOU KNOW? Chocolate improves brain function Banana reduces depression Orange improves skin health Cucumber keeps you hydrated FITTYWITTYOFFICIAL DID YOU KNOW? Clicn Aula Inimov bulin Iutktken Lunann roducc; dcorosuion Otnon ITtcrarit LuguTte Knutevounydruleo'
Prediction: Non-Bullying (Confidence: 0.0002)

Image: 5512.jpg | Text: 'DID YOU KNOW? Chocolate improves brain f...'
Model Said: Non-Bullying | True Img Label: 0 | True Text Label: 0



Predicting: 100%|██████████| 20/20 [00:06<00:00,  2.90it/s]

Input Text: 'modiji jo america ko dia wo hame bhi do na kya?wo hydrogen are hydroxychloroquine hai wo gand me hydrogen bharo re iske'
Image Path: 'D:/THE_CYBERBULLY_SHIELD/Datasets/multimodal-cyberbullying/bully_data\4578.jpg'
Processed Combined Text: 'modiji jo america ko dia wo hame bhi do na kya?wo hydrogen are hydroxychloroquine hai wo gand me hydrogen bharo re iske Modiji Fnnnen Wo hamc bhido n? Kya Wo Hydrogen Hydroxychloroquine hai wo (eandene tdcoren Onan Iske'
Prediction: Bullying (Confidence: 0.9756)

Image: 4578.jpg | Text: 'modiji jo america ko dia wo hame bhi do ...'
Model Said: Bullying | True Img Label: 1 | True Text Label: 1


FINISHED! Tested 20 valid image samples.
Accuracy vs Img-Label:  95.00%
Accuracy vs Text-Label: 75.00%
